In [9]:
# ============================================================
# PLAGIARISM DETECTION USING TF-IDF AND COSINE SIMILARITY
# ============================================================

import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# 1. LOAD DATASET
# ============================================================

file_path = "ASAP2_train_sourcetexts.csv"

df = pd.read_csv(
    file_path,
    engine="python",
    on_bad_lines="skip"
)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print(df.columns.tolist())


# ============================================================
# 2. FIND TEXT COLUMN
# ============================================================

# Try common text-column names first

possible_columns = [
    "full_text",
    "text",
    "essay",
    "essay_text",
    "source_text",
    "content"
]

text_column = None

for column in possible_columns:

    if column in df.columns:
        text_column = column
        break


# If no common name is found,
# automatically select the column containing the longest text

if text_column is None:

    string_columns = df.select_dtypes(
        include=["object"]
    ).columns

    if len(string_columns) == 0:
        raise ValueError("No text column found in the dataset.")

    average_lengths = {}

    for column in string_columns:

        average_lengths[column] = (
            df[column]
            .fillna("")
            .astype(str)
            .str.len()
            .mean()
        )

    text_column = max(
        average_lengths,
        key=average_lengths.get
    )


print("\nText column selected:", text_column)


# ============================================================
# 3. REMOVE EMPTY TEXT
# ============================================================

df[text_column] = (
    df[text_column]
    .fillna("")
    .astype(str)
)

df = df[
    df[text_column].str.strip() != ""
].reset_index(drop=True)


print("Documents available:", len(df))


# ============================================================
# 4. CLEAN AND NORMALIZE TEXT
# ============================================================

def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+",
        " ",
        text
    )

    # Remove punctuation and numbers
    text = re.sub(
        r"[^a-z\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


df["cleaned_text"] = df[text_column].apply(
    clean_text
)


print("\nText cleaning completed.")


# ============================================================
# 5. TOKENIZATION
# ============================================================

def tokenize(text):

    return text.split()


df["tokens"] = df["cleaned_text"].apply(
    tokenize
)


print("\nTokenization completed.")

print("\nExample tokens:")
print(df["tokens"].iloc[0][:20])


# ============================================================
# 6. CONVERT TEXT INTO TF-IDF VECTORS
# ============================================================

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=10000
)

tfidf_matrix = vectorizer.fit_transform(
    df["cleaned_text"]
)


print("\nTF-IDF conversion completed.")

print(
    "TF-IDF matrix shape:",
    tfidf_matrix.shape
)


# ============================================================
# 7. CALCULATE COSINE SIMILARITY
# ============================================================

similarity_matrix = cosine_similarity(
    tfidf_matrix
)


print("\nCosine similarity calculated.")


# ============================================================
# 8. DISPLAY SIMILARITY MATRIX
# ============================================================

# Use first 10 documents for easy display

number_to_display = min(
    10,
    len(df)
)

display_matrix = similarity_matrix[
    :number_to_display,
    :number_to_display
]

print("\nSimilarity Matrix:")
print(
    np.round(
        display_matrix,
        2
    )
)


# ============================================================
# 9. SET PLAGIARISM THRESHOLD
# ============================================================

threshold = 0.70


print("\nPlagiarism threshold:",
      threshold * 100,
      "%")


# ============================================================
# 10. COMPARE DOCUMENT PAIRS
# ============================================================

pairs = []

for i in range(len(df)):

    for j in range(i + 1, len(df)):

        score = similarity_matrix[i][j]

        percentage = score * 100

        if score >= threshold:

            status = "Possible Plagiarism"

        else:

            status = "Low Similarity"

        pairs.append({
            "Document 1": i + 1,
            "Document 2": j + 1,
            "Similarity (%)": percentage,
            "Status": status
        })


# ============================================================
# 11. SORT BY SIMILARITY
# ============================================================

pairs_df = pd.DataFrame(pairs)

pairs_df = pairs_df.sort_values(
    by="Similarity (%)",
    ascending=False
).reset_index(drop=True)


# ============================================================
# 12. DISPLAY TOP SUSPICIOUS PAIRS
# ============================================================

print("\n")
print("=" * 70)
print("        RANKED PLAGIARISM REPORT")
print("=" * 70)

for index, row in pairs_df.head(20).iterrows():

    doc1 = int(row["Document 1"])
    doc2 = int(row["Document 2"])

    score = row["Similarity (%)"]
    status = row["Status"]

    print(
        f"{index + 1}. "
        f"Assignment {doc1} vs Assignment {doc2}"
    )

    print(
        f"   Similarity: {score:.2f}%"
    )

    print(
        f"   Status: {status}"
    )

    print("-" * 70)


# ============================================================
# 13. DISPLAY ONLY SUSPICIOUS PAIRS
# ============================================================

suspicious_pairs = pairs_df[
    pairs_df["Similarity (%)"] >= threshold * 100
]


print("\n")
print("=" * 70)
print("        SUSPICIOUS DOCUMENT PAIRS")
print("=" * 70)


if len(suspicious_pairs) == 0:

    print("No document pairs exceeded the threshold.")

else:

    for index, row in suspicious_pairs.iterrows():

        doc1 = int(row["Document 1"])
        doc2 = int(row["Document 2"])

        score = row["Similarity (%)"]

        print(
            f"Assignment {doc1} vs Assignment {doc2}"
        )

        print(
            f"Similarity: {score:.2f}%"
        )

        print(
            "Possible Plagiarism"
        )

        print("-" * 70)


# ============================================================
# 14. SAVE REPORT TO CSV
# ============================================================

pairs_df.to_csv(
    "plagiarism_similarity_report.csv",
    index=False
)

print("\nReport saved as:")
print("plagiarism_similarity_report.csv")


# ============================================================
# 15. SAVE SUSPICIOUS PAIRS
# ============================================================

suspicious_pairs.to_csv(
    "suspicious_pairs.csv",
    index=False
)

print("Suspicious pairs saved as:")
print("suspicious_pairs.csv")

Dataset loaded successfully!
Dataset shape: (7476, 14)
['essay_id', 'score', 'full_text', 'assignment', 'prompt_name', 'economically_disadvantaged', 'student_disability_status', 'ell_status', 'race_ethnicity', 'gender', 'source_text_1', 'source_text_2', 'source_text_3', 'source_text_4']

Text column selected: full_text
Documents available: 7476

Text cleaning completed.

Tokenization completed.

Example tokens:
['the', 'author', 'suggests', 'that', 'studying', 'venus', 'is', 'worthy', 'enough', 'even', 'though', 'it', 'is', 'very', 'dangerous', 'the', 'author', 'mentioned', 'that', 'on']

TF-IDF conversion completed.
TF-IDF matrix shape: (7476, 10000)


MemoryError: Unable to allocate 425. MiB for an array with shape (55657254,) and data type float64